In [7]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment("BankGuard-Fraud-Detection")

print("MLflow Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name("BankGuard-Fraud-Detection").name)

MLflow Tracking URI: http://127.0.0.1:5000
Experiment: BankGuard-Fraud-Detection


#### Load your existing BankGuard model

In [8]:
from pathlib import Path
import joblib
import numpy as np

PROJECT_ROOT = Path.cwd().parents[1]

MODEL_PATH =(
    PROJECT_ROOT
    / "ml"
    / "artifacts"
    / "models"
    / "fraud_xgboost_engineered.pkl"
)

PROCESSED_DIR = (
    PROJECT_ROOT 
    / "ml"
    / "data"
    / "processed"
)

model = joblib.load(MODEL_PATH)

X_train = np.load(PROCESSED_DIR / "X_train_engineered.npy")
X_test = np.load(PROCESSED_DIR / "X_test_engineered.npy")
y_train = np.load(PROCESSED_DIR / "y_train_engineered.npy")
y_test = np.load(PROCESSED_DIR / "y_test_engineered.npy")

print("Model loaded successfully")
print("Model type:", type(model).__name__)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


Model loaded successfully
Model type: XGBClassifier
X_train: (5090096, 21)
X_test : (1272524, 21)
y_train: (5090096,)
y_test : (1272524,)


##### Calculate the evaluation metrics

In [11]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("Precision :", precision)
print("Recall    :", recall)
print("F1-score  :", f1)
print("ROC-AUC   :", roc_auc)
print("PR-AUC    :", pr_auc)

print("\nConfusion Matrix:")
print(cm)

Precision : 0.9873493975903614
Recall    : 0.9975654290931223
F1-score  : 0.992431123221314
ROC-AUC   : 0.9997474375086445
PR-AUC    : 0.9986538753959054

Confusion Matrix:
[[1270860      21]
 [      4    1639]]


#### create the first MLflow run

Use the metrics you have already calculated:

In [13]:
with mlflow.start_run(run_name="XGBoost-Engineered-Features"):

    # Model information
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("feature_pipeline", "Engineered Features")
    mlflow.log_param("feature_count", model.n_features_in_)

    # Dataset information
    mlflow.log_param("dataset", "PaySim")
    mlflow.log_param("target", "isFraud")
    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("test_samples", X_test.shape[0])

    # XGBoost parameters
    mlflow.log_param("n_estimators", model.n_estimators)
    mlflow.log_param("max_depth", model.max_depth)
    mlflow.log_param("learning_rate", model.learning_rate)
    mlflow.log_param("subsample", model.subsample)
    mlflow.log_param("colsample_bytree", model.colsample_bytree)
    mlflow.log_param("scale_pos_weight", model.scale_pos_weight)

    # Metrics
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)

    print("MLflow run created successfully.")

    run_id = mlflow.active_run().info.run_id
    print("Run ID:", run_id)

MLflow run created successfully.
Run ID: c9d6a89bbac24a0d94361b254fa23f03
🏃 View run XGBoost-Engineered-Features at: http://127.0.0.1:5000/#/experiments/1/runs/c9d6a89bbac24a0d94361b254fa23f03
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


##### log the XGBoost model

In [14]:
with mlflow.start_run(run_name="XGBoost-Engineered-Features"):

    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("feature_pipeline", "Engineered Features")
    mlflow.log_param("feature_count", model.n_features_in_)

    mlflow.log_param("dataset", "PaySim")
    mlflow.log_param("target", "isFraud")
    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("test_samples", X_test.shape[0])

    mlflow.log_param("n_estimators", model.n_estimators)
    mlflow.log_param("max_depth", model.max_depth)
    mlflow.log_param("learning_rate", model.learning_rate)
    mlflow.log_param("subsample", model.subsample)
    mlflow.log_param("colsample_bytree", model.colsample_bytree)
    mlflow.log_param("scale_pos_weight", model.scale_pos_weight)

    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)

    mlflow.xgboost.log_model(
        model,
        name="fraud_xgboost_model",
        model_format="json"
    )

    print("Model logged successfully.")

Model logged successfully.
🏃 View run XGBoost-Engineered-Features at: http://127.0.0.1:5000/#/experiments/1/runs/9ddafbf32b8942d3892a46ed2c14134e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


##### add the confusion matrix and model report

In [15]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

with mlflow.start_run(run_name="XGBoost-Engineered-Features-Full"):

    # Parameters
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("feature_pipeline", "Engineered Features")
    mlflow.log_param("feature_count", model.n_features_in_)

    mlflow.log_param("dataset", "PaySim")
    mlflow.log_param("target", "isFraud")
    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("test_samples", X_test.shape[0])

    mlflow.log_param("n_estimators", model.n_estimators)
    mlflow.log_param("max_depth", model.max_depth)
    mlflow.log_param("learning_rate", model.learning_rate)
    mlflow.log_param("subsample", model.subsample)
    mlflow.log_param("colsample_bytree", model.colsample_bytree)
    mlflow.log_param("scale_pos_weight", model.scale_pos_weight)

    # Metrics
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(6, 6))

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Legitimate", "Fraud"]
    ).plot(ax=ax)

    ax.set_title("BankGuard XGBoost Confusion Matrix")
    plt.tight_layout()

    confusion_matrix_path = "../artifacts/reports/xgboost_confusion_matrix.png"

    Path("../artifacts/reports").mkdir(
        parents=True,
        exist_ok=True
    )

    plt.savefig(confusion_matrix_path, dpi=150)
    plt.close()

    # Log confusion matrix
    mlflow.log_artifact(confusion_matrix_path)

    # Log model
    mlflow.xgboost.log_model(
        model,
        name="fraud_xgboost_model",
        model_format="json"
    )

    print("Full MLflow run completed.")
    print("Run ID:", mlflow.active_run().info.run_id)

Full MLflow run completed.
Run ID: 87cd176aee214bc898aacabedb0889d3
🏃 View run XGBoost-Engineered-Features-Full at: http://127.0.0.1:5000/#/experiments/1/runs/87cd176aee214bc898aacabedb0889d3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


##### Register the model

Use the run ID from your successful model-logging cell.

In [16]:
import mlflow

MODEL_NAME = "BankGuard-Fraud-XGBoost"

run_id = "9ddafbf32b8942d3892a46ed2c14134e"

model_uri = f"runs:/{run_id}/fraud_xgboost_model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=MODEL_NAME
)

print("Registered model:", registered_model.name)
print("Version:", registered_model.version)

Successfully registered model 'BankGuard-Fraud-XGBoost'.
2026/10/03 18:52:43 WARNING mlflow.tracking._model_registry.fluent: Run with id 9ddafbf32b8942d3892a46ed2c14134e has no artifacts at artifact path 'fraud_xgboost_model', registering model based on models:/m-67c9078047a54534a3906cf01490dc8f instead
2026/10/03 18:52:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: BankGuard-Fraud-XGBoost, version 1


Registered model: BankGuard-Fraud-XGBoost
Version: 1


Created version '1' of model 'BankGuard-Fraud-XGBoost'.


##### Give Version 1 an alias

Current MLflow uses aliases for mutable references to model versions.

In [17]:
from mlflow import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name="BankGuard-Fraud-XGBoost",
    alias="production",
    version="1"
)

print("Production alias assigned to Version 1")

Production alias assigned to Version 1


##### Verify the production model

In [18]:
model_version = client.get_model_version_by_alias(
    "BankGuard-Fraud-XGBoost",
    "production"
)

print("Model:", model_version.name)
print("Version:", model_version.version)
print("Alias: production")

Model: BankGuard-Fraud-XGBoost
Version: 1
Alias: production


##### Test loading the registered production model

##### This is important because it proves that the application can later load the model through MLflow, rather than depending directly on:

In [19]:
production_model = mlflow.xgboost.load_model(
    "models:/BankGuard-Fraud-XGBoost@production"
)

print("Production model loaded successfully.")
print("Model type:", type(production_model).__name__)

Production model loaded successfully.
Model type: XGBClassifier


In [20]:
print("Features expected:", production_model.n_features_in_)

Features expected: 21
